# LoRA/PEFT 로 LLM 효율적 미세조정 (SFT) — Disaster Tweets — Colab

전체 가중치 대신 **소수의 LoRA 어댑터만** 학습하는 **파라미터 효율 미세조정(PEFT)** 기본 예제입니다.

- 핵심 흐름: 사전학습 LM에 **LoRA 어댑터** 삽입 → **학습 파라미터 1% 미만**만 업데이트 → 생성.
- IOAI 2025 **Concepts** 과제의 기반 기법입니다. (공식 해법: `unsloth FastLanguageModel` + `trl SFT` = LoRA 기반)
  이식성을 위해 표준 `peft`+`transformers` 로 동일 개념(LoRA)을 연습합니다. 16번(전체 미세조정)과 비교해 보세요.
- [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) 트윗 코퍼스 사용. 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ GPU 권장. ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "transformers", "peft", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 트윗 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료")

## 2. 모델 로드 & 코퍼스 토크나이즈

In [ ]:
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "distilgpt2"
tok = AutoTokenizer.from_pretrained(MODEL); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL)

texts = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))["text"].fillna("").tolist()
ids = tok((" "+tok.eos_token+" ").join(texts), return_tensors="pt")["input_ids"][0]
BLOCK = 64; nb = ids.size(0)//BLOCK; blocks = ids[:nb*BLOCK].view(nb, BLOCK)
print("blocks:", tuple(blocks.shape))

## 3. LoRA 어댑터 삽입 (PEFT)

GPT-2 의 어텐션 projection(`c_attn`)에 저랭크 어댑터를 붙입니다. **학습 파라미터가 전체의 1% 미만**으로 줄어듭니다.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05,
                  target_modules=["c_attn"])
model = get_peft_model(base, lora)

tot = sum(p.numel() for p in model.parameters())
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 파라미터 {tot:,} | 학습(LoRA) 파라미터 {tr:,} ({100*tr/tot:.2f}%)")
model.print_trainable_parameters()

## 4. LoRA 미세조정 (어댑터만 학습)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.to(device)
loader = DataLoader(TensorDataset(blocks), batch_size=16, shuffle=True)
opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)

for epoch in range(1, 4):
    model.train(); run=0; n=0
    for (b,) in loader:
        b = b.to(device)
        out = model(input_ids=b, labels=b)
        out.loss.backward(); opt.step(); opt.zero_grad()
        run += out.loss.item()*b.size(0); n += b.size(0)
    print(f"epoch {epoch} | LM loss {run/n:.3f}")

## 5. 생성

In [ ]:
model.eval()
for p in ["Breaking news:", "The storm"]:
    ii = tok(p, return_tensors="pt").input_ids.to(device)
    out = model.generate(ii, max_new_tokens=35, do_sample=True, top_k=50, top_p=0.95,
                         temperature=0.9, pad_token_id=tok.eos_token_id)
    print("•", tok.decode(out[0], skip_special_tokens=True).replace("\n"," "))
    print()

## 🏁 제출 안내

이 노트북은 **LoRA/PEFT 효율 미세조정** 연습용 데모입니다(제출 리더보드 없음).

- 코퍼스 출처: **[NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started)**

> IOAI 2025 *Concepts*(unsloth+trl SFT=LoRA) 의 기반 기법입니다. 16번(전체 미세조정) 대비 **학습 파라미터를 100배 이상 절감**하면서 적응하는 현대 표준 기법입니다.